In [1]:
import pandas as pd
df = pd.read_csv("507_traditional_dataset.csv")
#specific lable classification
df["LABEL"].unique()

array(['neutral', 'slightly_suspicious', 'suspicious',
       'highly_suspicious', 'legitimate', 'scam',
       'standard_opening, identification_request', 'polite_ending',
       'potential_scam', ' neutral', ' legitimate', ' scam',
       ' scam_response', ' dismissing official protocols"',
       ' emphasizing security and compliance"',
       ' ready for further engagement"',
       ' suggesting a dangerous situation"', 'Scam',
       ' adhering to protocols"', ' citing urgency"', 'scam_response'],
      dtype=object)

In [2]:
#data cleaning(make sure string, remove the space and residual symbols, unify the lowercase)
df["LABEL_CLEAN"] = (df["LABEL"]
    .astype(str)           
    .str.strip()           
    .str.lower()           
    .str.replace('"', ''))  
df["LABEL_CLEAN"].unique()


array(['neutral', 'slightly_suspicious', 'suspicious',
       'highly_suspicious', 'legitimate', 'scam',
       'standard_opening, identification_request', 'polite_ending',
       'potential_scam', 'scam_response', 'dismissing official protocols',
       'emphasizing security and compliance',
       'ready for further engagement', 'suggesting a dangerous situation',
       'adhering to protocols', 'citing urgency'], dtype=object)

In [3]:
#classify(scam vs non-scam)
scam_keywords = [
    "scam",                # scam, potential_scam, scam_response 
    "suspicious",          # slightly_suspicious / highly_suspicious
    "dangerous",           # suggesting a dangerous situation
    "urgency",             # citing urgency
    "dismissing",          # dismissing official protocols
    "compliance",          # emphasizing security and compliance
    "ready for further engagement"]


In [4]:
#mapping fuction
def map_label_to_class(label_clean: str):
    for kw in scam_keywords:
        if kw in label_clean:
            return 1, "traditional_scam"
    return 0, "non_scam"
df[["label_id", "label_text"]] = df["LABEL_CLEAN"].apply(
    lambda x: pd.Series(map_label_to_class(x))
)
df["label_text"].value_counts()

label_text
traditional_scam    397
non_scam            253
Name: count, dtype: int64

In [5]:
#build a standardized training DataFrame
df_std = pd.DataFrame({
    "id": range(len(df)),
    "text": df["TEXT"],             
    "label_id": df["label_id"],
    "label_text": df["label_text"],
    "source": "Kaggle_Call Transcripts Scam Determinations",  
    "is_synthetic": 0})

#new file
output_path = "traditional_calls_standardized.csv"
df_std.to_csv(output_path, index=False)
output_path

'traditional_calls_standardized.csv'

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True)
model.to("cpu")

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [17]:
def generate_ai_scam_text(scenario: str) -> str:
    """
    Use Qwen2.5-1.5B-Instruct to generate one AI-style scam message.
    Output only the scam message itself.
    """
    prompt = (
    f"Scenario: {scenario}\n"
    "Generate a short scam message spoken by a phone caller.\n"
    "Make it sound like real spoken language: slightly rushed, informal, and a bit hesitant.\n"
    "Avoid email style, avoid paragraphs, avoid HTML, avoid <br>.\n"
    "Do NOT ask for money directly. Instead, talk around the issue and hint at urgency.\n"
    "Use fillers like 'uh', 'listen', 'so', 'I mean', to imitate phone speech.\n"
    "Keep it short: 20–45 words.\n"
    "Use placeholders like <NAME>, <BANK>, <ACCOUNT>.\n"
    "Output ONLY the spoken message.\n"
    "\nMessage:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,)
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Message:" in full_text:
        return full_text.split("Message:")[-1].strip()
    else:
        return full_text.strip()


In [19]:
#test
print(generate_ai_scam_text(
    "AI-written bank notification saying that the user's account is temporarily locked and asking them to verify their identity via a fake online link."
))

[Caller starts speaking hesitantly] "Listen, uh... I got your account flagged as suspicious or something. So, um, could you please verify your ID? It’s super important to keep your stuff safe. But wait! Don’t worry, I’m just trying to help, okay?


In [20]:
#setting scenarios
ai_scam_scenarios = [
    # 1. AI-generated voice spoofing a family member
    "AI voice deepfake pretending to be a family member who claims to be in urgent trouble and needs money immediately.",
    # 2. AI-written bank fraud notification
    "AI-written bank notification saying that the user's account is temporarily locked and asking them to verify their identity via a fake online link.",
    # 3. AI-generated message impersonating delivery or logistics support
    "AI-generated message impersonating a delivery or shipping company, asking the user to pay an extra customs or service fee through a fake payment page.",
    # 4. AI-generated email pretending to be a school or immigration office
    "AI-generated email pretending to be from a university or immigration office, asking the user to upload sensitive personal documents to a fraudulent website.",
    # 5. AI-generated investment scam
    "AI-generated investment scam promising unusually high returns with low risk, asking the user to transfer money to a so-called secure investment account."
]

In [21]:
from tqdm import tqdm
import pandas as pd
rows = []
num_per_scenario = 10
for scenario in tqdm(ai_scam_scenarios, desc="Generating by Scenario"):
    for i in tqdm(range(num_per_scenario), desc=f"{scenario[:25]}..", leave=False):
        text = generate_ai_scam_text(scenario)
        rows.append({
            "text": text,
            "label_id": 2,
            "label_text": "ai_scam",
            "source": "qwen_generated",
            "scenario": scenario,
            "is_synthetic": 1})
df_ai = pd.DataFrame(rows)
df_ai.head(), df_ai["label_text"].value_counts()

Generating by Scenario:   0%|                             | 0/5 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

Generating by Scenario: 100%|█████████████████████| 5/5 [06:33<00:00, 78.67s/it]


(                                                text  label_id label_text  \
 0  Hello, my name is [NAME], but I need your help...         2    ai_scam   
 1  Listen, I'm uh... my brother... he's in uh... ...         2    ai_scam   
 2  Listen, uh... I'm <NAME> from [NAME], we're fa...         2    ai_scam   
 3  "Listen, uh, my mommy's in uh... uh... she's u...         2    ai_scam   
 4                  "Hello there, is this [NAME]? Uh,         2    ai_scam   
 
            source                                           scenario  \
 0  qwen_generated  AI voice deepfake pretending to be a family me...   
 1  qwen_generated  AI voice deepfake pretending to be a family me...   
 2  qwen_generated  AI voice deepfake pretending to be a family me...   
 3  qwen_generated  AI voice deepfake pretending to be a family me...   
 4  qwen_generated  AI voice deepfake pretending to be a family me...   
 
    is_synthetic  
 0             1  
 1             1  
 2             1  
 3            

In [22]:
ai_path = "ai_scam_qwen_generated.csv"
df_ai.to_csv(ai_path, index=False)
ai_path

'ai_scam_qwen_generated.csv'

In [23]:
df_trad = pd.read_csv("traditional_calls_standardized.csv")
df_trad.head()
# add id for AI data
df_ai_std = df_ai.reset_index().rename(columns={"index": "id"})
df_ai_std = df_ai_std[["id", "text", "label_id", "label_text", "source", "is_synthetic"]]
#
df_trad_std = df_trad[["id", "text", "label_id", "label_text", "source", "is_synthetic"]]

# combine:non_scam + traditional_scam）+AI scam
df_all = pd.concat([df_trad_std, df_ai_std], ignore_index=True)

df_all["label_text"].value_counts(), df_all.head()

(label_text
 traditional_scam    397
 non_scam            253
 ai_scam              50
 Name: count, dtype: int64,
    id                                               text  label_id label_text  \
 0   0  Good morning, this is [Your Name]'s personal a...         0   non_scam   
 1   1  Hello, my name is Jamie. I'm interested in vol...         0   non_scam   
 2   2  Yes, I'm really passionate about environmental...         0   non_scam   
 3   3  Great, how do I sign up, and where can I find ...         0   non_scam   
 4   4  Could you send me the link, please? And my ema...         0   non_scam   
 
                                         source  is_synthetic  
 0  Kaggle_Call Transcripts Scam Determinations             0  
 1  Kaggle_Call Transcripts Scam Determinations             0  
 2  Kaggle_Call Transcripts Scam Determinations             0  
 3  Kaggle_Call Transcripts Scam Determinations             0  
 4  Kaggle_Call Transcripts Scam Determinations             0  )

In [24]:
full_path = "call_scam_dataset_traditional_and_ai_qwen.csv"
df_all.to_csv(full_path, index=False)
full_path

'call_scam_dataset_traditional_and_ai_qwen.csv'